In [1]:
#from graph_transformer_long_range_niches.tl.wandb import load_and_log
from graph_transformer_long_range_niches._paths import CFG_FILES, HE22_HUMAN_LUNG_DATA_PATH
from graph_transformer_long_range_niches.model.gnn_transformer import LitGNNTransformer
from graph_transformer_long_range_niches.modules.gcn import LitGCN
from graph_transformer_long_range_niches.tl.utils import pad_batch
from graph_transformer_long_range_niches.tl.evaluation import eval_gnntransformer, extract_attention
from graph_transformer_long_range_niches.pl.color_map import CustomColormap
from graph_transformer_long_range_niches.pp.geome_utils import prepare_geome_dataset
from graph_transformer_long_range_niches.config import load_config
from graph_transformer_long_range_niches.pl.attention_matrix import plot_attention_sender_receiver, SelfAttentionRelevance

import warnings
warnings.filterwarnings("ignore")

from torch_geometric.loader import DataLoader

from pathlib import Path
import wandb
import torch
import os

## plotting
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import scanpy as sc
import squidpy as sq

import pickle

# Load model checkpoint and data

In [2]:
#cfg_trans = load_config(Path(CFG_FILES, 'pancreas_gnntrans_genes.yaml'))
cfg_trans = load_config(Path(CFG_FILES, 'pancreas_gnntrans_ct.yaml'))
cfg_gnn = load_config(Path(CFG_FILES, 'pancreas_gnn_genes.yaml'))

In [3]:
data_gnntrans = f"{cfg_trans.dataset.name}_{cfg_trans.dataset.prediction_obs}_{cfg_trans.dataset.library_key}_{cfg_trans.optim.seed}"
run_gnntrans = f"{data_gnntrans}_{cfg_trans.model.model_type}"
print(data_gnntrans)
print(run_gnntrans)
data_gnn = f"{cfg_gnn.dataset.name}_{cfg_gnn.dataset.prediction_obs}_{cfg_gnn.dataset.library_key}_{cfg_gnn.optim.seed}"
run_gnn = f"{data_gnn}_{cfg_gnn.model.model_type}"
print(data_gnn)
print(run_gnn)

pancreas_cell_type_coarse_sliding_window_40
pancreas_cell_type_coarse_sliding_window_40_gnn-transformer
pancreas_condition_sliding_window_42
pancreas_condition_sliding_window_42_gnn


In [4]:
checkpoint = torch.load(Path(cfg_trans.model.output_path, f"{run_gnntrans}.ckpt"))
model_transformer = LitGNNTransformer(checkpoint['hyper_parameters']['cfg'])
model_transformer.load_state_dict(checkpoint['state_dict'])

<All keys matched successfully>

In [5]:
# checkpoint_gnn = torch.load(Path(cfg_gnn.model.output_path, f"{run_gnn}.ckpt"))
# model_gnn = LitGCN(checkpoint_gnn['hyper_parameters']['cfg'])
# model_gnn.load_state_dict(checkpoint_gnn['state_dict'])

In [6]:
adata = sc.read_h5ad(Path(cfg_trans.model.output_path, f"{data_gnntrans}.h5ad"))
adata

AnnData object with n_obs × n_vars = 104816 × 979
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.GCG', 'Max.GCG', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'cell_ID', 'condition', 'slide', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_NegPrb', 'log1p_total_counts_NegPrb', 'pct_counts_NegPrb', 'n_genes', 'cell_type_coarse', 'niche_label', 'x', 'y', 'sliding_window', 'split'
    uns: 'cell_type_coarse_colors', 'sliding_window_colors', 'spatial', 'spatial_neighbors'
    obsm: 'spatial', 'spatial_fov'
    layers: 'counts'

In [7]:
with open(Path(cfg_trans.model.output_path, f"{data_gnntrans}.pkl"), 'rb') as f:
    pyg_data = pickle.load(f)
train_pyg = pyg_data[0]
val_pyg = pyg_data[1]
print('train: ', len(train_pyg), 'val: ', len(val_pyg))

train:  108 val:  12


In [8]:
train_adata = adata[adata.obs['split'] == 'train']
val_adata = adata[adata.obs['split'] == 'val']

In [9]:
print(np.unique(train_adata.obs['fov']))
print(np.unique(val_adata.obs['fov']))

['1' '10' '11' '12' '14' '15' '16' '18' '2' '20' '21' '22' '23' '3' '4'
 '7' '8' '9']
['17' '6']


# Attention Relevance

How relevant is a node for a specific class prediction?

In [10]:
sel_attention_relevance = SelfAttentionRelevance(model_transformer.transformer_encoder)

In [11]:
start, end = 0,1 # FOV: 12
eval_loader = DataLoader(val_pyg[start:end], 1)
for i, batch in enumerate(eval_loader):
    transformer_in, transformer_out, src_padding_mask, index_nodes, dec_out = model_transformer.evaluation(batch)

In [12]:
I = sel_attention_relevance.generate_relevance(transformer_in, src_padding_mask, category_index=index_nodes[0])

Category mask:  torch.Size([994, 1, 128])
994
Layer 1 attention gradient shape: torch.Size([4, 994, 994])
Layer 1 attention output weights shape: torch.Size([4, 994, 994])
Average attention map shape: torch.Size([994, 994])
I:  tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [0., 0., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.]], device='cuda:0')


In [16]:
for window_id in np.unique(val_adata.obs['sliding_window']):
    

17_0_0
17_0_1
17_1_0
17_1_1
17_2_0
17_2_1
6_0_0
6_0_1
6_1_0
6_1_1
6_2_0
6_2_1
